<a href="https://colab.research.google.com/github/AustinChinn03/DS3001Final/blob/main/DataScientist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Demand Estimation

We will estimate a logit-style demand model using linear regression. The model is:

$$
\log(s_{bt}) = \alpha_0 + \alpha_t + \gamma_b + \beta_{price}p_{bt} + \beta_{rating}r_{bt} + \sum_{\ell=1}^L \beta_\ell x_{bt\ell} + \epsilon_{bt}.
$$

Here:

- $b$ indexes brands
- $t$ indexes years
- $s_{bt}$ is `brand_share`
- $p_{bt}$ is `avg_price`
- $r_{bt}$ is `avg_rating`
- $x_{bt\ell}$ are the product characteristics
- $\alpha_t$ are year dummy coefficients
- $\gamma_b$ are brand dummy coefficients
- $\beta_{price}$ is **one constant price coefficient**, shared by all brands and all years

That last point matters: do **not** estimate a different price coefficient for every brand-year. We do not have enough information for that, and it would make the cost calculation impossible to interpret.

Use `pd.get_dummies(..., drop_first=True)` for brand and year dummies. The dropped brand and dropped year become the reference categories, so all dummy coefficients are interpreted relative to those omitted categories.

Questions:

1. What is the estimated price coefficient, $\hat{\beta}_{price}$?
2. Is it negative? Why is that important?
3. Which product features are associated with higher demand?
4. Which brand dummy coefficients are largest? Remember that these are interpreted relative to the dropped brand.
5. Which year dummy coefficients are largest? Remember that these are interpreted relative to the dropped year.
6. What is the model's $R^2$?

This part of the work is the **data scientist** role: turning the cleaned data into a model that can be used for prediction and interpretation.

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
!git clone https://github.com/AustinChinn03/DS3001Final.git

fatal: destination path 'DS3001Final' already exists and is not an empty directory.


In [16]:
air_fryers = pd.read_csv("DS3001Final/air_fryers_clean_brand_year.csv")
air_fryers

,category,year,brand,purchase_count,product_count,avg_price,avg_rating,compact_share,dual_basket_share,oven_style_share,rotisserie_share,window_share,market_purchases,brand_share,log_brand_share
0,air_fryers,2019,chefman,1146,10,72.963695,4.434119,1.000000,0.000000,0.780977,0.243455,0.184119,15076,0.076015,-2.576826
1,air_fryers,2019,cosori,11,2,159.990000,4.581818,1.000000,0.000000,0.090909,0.090909,0.000000,15076,0.000730,-7.222964
2,air_fryers,2019,cuisinart,1616,22,229.465274,4.481312,0.993812,0.000000,0.889851,0.000000,0.000000,15076,0.107190,-2.233150
3,air_fryers,2019,dash,3011,19,55.176333,4.390767,1.000000,0.000000,0.973431,0.000000,0.000000,15076,0.199721,-1.610832
4,air_fryers,2019,gowise usa,4405,45,83.575551,4.552259,0.999773,0.000000,0.129398,0.128490,0.000000,15076,0.292186,-1.230364
5,air_fryers,2019,instant_pot,238,13,78.019945,4.364286,0.542017,0.000000,0.550420,0.000000,0.000000,15076,0.015787,-4.148589
6,air_fryers,2019,ninja,2904,18,112.158374,4.778650,0.994490,0.000000,0.000000,0.000000,0.000000,15076,0.192624,-1.647015
7,air_fryers,2019,nuwave,484,15,151.099937,4.321901,0.993802,0.000000,0.561983,0.111570,0.000000,15076,0.032104,-3.438774
8,air_fryers,2019,oster,506,5,191.943688,4.380830,1.000000,0.000000,0.772727,0.000000,0.000000,15076,0.033563,-3.394323
9,air_fryers,2019,ultrean,755,4,80.624675,4.618013,1.000000,0.000000,0.835762,0.000000,0.000000,15076,0.050080,-2.994142


In [17]:
char_cols = [
    'compact_share', 'dual_basket_share', 'oven_style_share',
    'rotisserie_share', 'window_share'
]
brand_dummies = pd.get_dummies(air_fryers['brand'], prefix='brand', drop_first=True)
year_dummies  = pd.get_dummies(air_fryers['year'].astype(str), prefix='year', drop_first=True)

In [18]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
X = pd.concat(
    [air_fryers[['avg_price', 'avg_rating'] + char_cols], brand_dummies, year_dummies],
    axis=1
).astype(float)

y = air_fryers['log_brand_share'].astype(float)

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)


In [19]:
coef = pd.Series(model.coef_, index=X.columns)
coef.sort_values(ascending=False)
price_coef = coef['avg_price']
print(f"Estimated price coefficient: {price_coef}")



Estimated price coefficient: -0.03766765298429383


In [20]:
for feature in char_cols:
  print(f"{feature}: {coef[feature]}")

compact_share: 9.815303774308129
dual_basket_share: -9.50968569808526
oven_style_share: 1.9417744851002332
rotisserie_share: -5.674054252708323
window_share: 12.880297890723153


In [21]:
brand_coefs = coef[coef.index.str.startswith('brand_')]
brand_coefs

,0
brand_cosori,2.551946
brand_cuisinart,6.422436
brand_dash,0.176655
brand_gowise usa,3.938996
brand_instant_pot,4.626260
brand_ninja,5.838705
brand_nuwave,3.544883
brand_oster,3.928074
brand_ultrean,0.942399


In [22]:
year_coefs = coef[coef.index.str.startswith('year_')]
year_coefs

,0
year_2020,0.119071
year_2021,0.041900
year_2022,-0.098860
year_2023,-0.003307


In [23]:
r2 = r2_score(y, y_pred)
print(f"R^2: {r2}")

R^2: 0.763453950091436


1. Price coefficient is -0.03766765298429383
2. It's negative which makes sense as price increases, the demand will decrease
3. compact, oven_style, and window are high demand
4. cuisinart is the largest
5. 2020 is the largest
6. R^2: 0.763453950091436